# 03b — Geração automática de Knowledge Components (KCGen-KT)

Pipeline de geração automática de KCs por assignment para o CSEDM, adaptado de Duan et al. (2025) — *KCGen-KT*. O fluxo completo (a ser implementado nas tarefas 1–7) compreende:

1. **Diversity sampling** de submissões corretas por problema (esta tarefa).
2. **Geração de KCs por LLM** a partir do código-fonte bruto (não AST — ablação de Duan et al., 2025, Table 4).
3. **Clustering Sentence-BERT + HAC** dos KCs brutos por assignment.
4. **Rotulagem dos clusters via LLM** (Table 9 de Duan et al.).
5. **Q-matrix per-assignment** (10–15 KCs).
6. **KC correctness labeling** das submissões incorretas via LLM (Table 10).
7. **Validação post-hoc com srcML** comparando KCs gerados a assinaturas AST.

**Decisões críticas (todas baseadas em Duan et al., 2025):**
- LLM recebe **código bruto** — NÃO AST (Table 4: AST-only piora AUC de 0.812 → 0.784).
- **5 submissões por problema** via stratified sampling por número de tentativas (Table 5 — n=5 é o ponto ótimo).
- Uso exclusivo de `Release/Train` (mesma população de Shi et al., 2022) para evitar data leakage.
- **Cache obrigatório** em todas as etapas LLM — notebook deve ser re-executável sem novas chamadas API.

**Modelo LLM:** `claude-haiku-4-5-20251001` (custo total estimado: ~\$39–40).

In [1]:
# Setup — imports, SEED, paths
import json
import pickle
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# Bibliotecas necessárias para as etapas posteriores do pipeline KCGen-KT
import anthropic                              # Etapas 2, 4 e 6 — chamadas LLM
import sentence_transformers                  # Etapa 3 — embeddings dos KCs
import scipy                                  # Etapa 3 — HAC clustering
import sklearn                                # Etapa 3 — métricas auxiliares

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = REPO_ROOT / "data" / "CSEDM"
RESULTS_ROOT = REPO_ROOT / "results"
RESULTS_ROOT.mkdir(exist_ok=True)

SEQUENCES_PATH = RESULTS_ROOT / "sequences_bkt_dkt.pkl"
CODE_STATES_PATH = DATA_ROOT / "Release" / "Train" / "Data" / "CodeStates" / "CodeStates.csv"

ASSIGNMENTS = [439, 487, 492, 494, 502]
LLM_MODEL = "claude-haiku-4-5-20251001"

print(f"SEED        = {SEED}")
print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"sequences   = {SEQUENCES_PATH.exists()} ({SEQUENCES_PATH.name})")
print(f"code states = {CODE_STATES_PATH.exists()} ({CODE_STATES_PATH.name})")
print(f"anthropic   = {anthropic.__version__}")
print(f"s-transformers = {sentence_transformers.__version__}")
print(f"scipy       = {scipy.__version__}")
print(f"sklearn     = {sklearn.__version__}")

/home/leokuntz/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SEED        = 42
REPO_ROOT   = /home/leokuntz/Documents/repositories/studies/tcc.edm.kt
sequences   = True (sequences_bkt_dkt.pkl)
code states = True (CodeStates.csv)
anthropic   = 0.99.0
s-transformers = 5.4.1
scipy       = 1.17.1
sklearn     = 1.8.0


### 1.1 — Carregamento e diversity sampling de submissões corretas

**Contexto:** A geração de KCs via LLM exige um conjunto pequeno e diverso de submissões corretas para cada problema. Como o CSEDM não fornece enunciados textuais dos problemas, o LLM deve inferir o conhecimento exigido a partir das soluções dos próprios estudantes — soluções demasiado parecidas levariam o modelo a sub-representar abordagens alternativas (e, portanto, KCs).

**Hipótese:** Estratificar as submissões corretas pelo número total de tentativas até o acerto (proxy para diversidade de estratégia: "acertou direto" vs "acertou após várias tentativas") permitirá amostrar até 5 submissões com cobertura ampla de abordagens, mantendo a fração ótima de Duan et al. (2025, Table 5) com custo proporcional.

**Referência:** Duan et al. (2025), *Automated Knowledge Component Generation for Interpretable Knowledge Tracing in Coding Problems* — Table 5 (ablação `n_submissions`: 1=0.798, 3=0.807, **5=0.812**, 7=0.811, 10=0.810). A estratificação por número de tentativas é a heurística adotada neste trabalho para garantir diversidade dentro do limite de n=5; Duan et al. amostram aleatoriamente, mas mencionam que a representatividade do conjunto importa para a qualidade dos KCs.

In [2]:
def load_correct_samples(sequences_path: Path, code_states_path: Path) -> dict:
    """Carrega submissões corretas do Release/Train com código Java associado.

    Para cada par (estudante, problema), identifica a primeira submissão correta
    cronologicamente, conta as tentativas anteriores (eventos do mesmo estudante
    no mesmo problema antes do acerto) e faz join com CodeStates.csv via
    CodeStateID para recuperar o código Java efetivamente submetido.

    Parameters
    ----------
    sequences_path : Path
        Caminho para results/sequences_bkt_dkt.pkl (artefato do notebook 02).
    code_states_path : Path
        Caminho para data/CSEDM/Release/Train/Data/CodeStates/CodeStates.csv.

    Returns
    -------
    dict
        {assignment_id: {problem_id: [
            {'subject_id': str, 'codestate_id': str, 'n_attempts_before': int,
             'total_attempts': int, 'code': str},
            ...
        ]}}
        Apenas a primeira submissão correta por (estudante, problema) é incluída.
    """
    with open(sequences_path, "rb") as f:
        artifact = pickle.load(f)

    code_states = pd.read_csv(code_states_path)
    code_map = dict(zip(code_states["CodeStateID"], code_states["Code"]))

    correct_by_assignment = {}
    for assignment_id, sequences in artifact["train"].items():
        per_problem: dict[int, list[dict]] = {}
        for seq in sequences:
            events = seq["events"].sort_values("ServerTimestamp", kind="stable")
            for problem_id, problem_events in events.groupby("ProblemID", sort=False):
                problem_events = problem_events.sort_values("ServerTimestamp", kind="stable")
                correct_mask = problem_events["correct"] == 1
                if not correct_mask.any():
                    continue
                first_correct_pos = correct_mask.values.argmax()
                first_correct = problem_events.iloc[first_correct_pos]
                code = code_map.get(first_correct["CodeStateID"])
                if code is None or (isinstance(code, float) and pd.isna(code)):
                    continue
                n_before = int(first_correct_pos)
                per_problem.setdefault(int(problem_id), []).append({
                    "subject_id": str(seq["subject_id"]),
                    "codestate_id": str(first_correct["CodeStateID"]),
                    "n_attempts_before": n_before,
                    "total_attempts": n_before + 1,
                    "code": str(code),
                })
        correct_by_assignment[int(assignment_id)] = per_problem

    return correct_by_assignment


def _bucket_for(total_attempts: int) -> int:
    """Mapeia número total de tentativas em um dos 5 buckets de diversidade.

    Buckets (Duan et al., 2025, motivação: diversidade de estratégia):
      1 → acertou na 1ª tentativa
      2 → 2–3 tentativas
      3 → 4–6 tentativas
      4 → 7–10 tentativas
      5 → >10 tentativas
    """
    if total_attempts <= 1:
        return 1
    if total_attempts <= 3:
        return 2
    if total_attempts <= 6:
        return 3
    if total_attempts <= 10:
        return 4
    return 5


def diversity_sample(correct_events: list[dict], n: int = 5,
                     rng: random.Random | None = None) -> list[dict]:
    """Estratifica as submissões corretas em 5 buckets e amostra até n.

    Para cada bucket disponível (na ordem 1..5), seleciona aleatoriamente uma
    submissão; retorna no máximo n=5 submissões. Buckets vazios são pulados.

    Parameters
    ----------
    correct_events : list[dict]
        Lista de submissões corretas para um par (assignment, problem) — cada
        item deve ter as chaves 'total_attempts' e 'code'.
    n : int
        Número máximo de submissões a retornar. Padrão 5 (Duan et al., 2025,
        Table 5 — ponto ótimo).
    rng : random.Random | None
        Gerador de aleatórios para reproduzibilidade. Se None, usa o gerador
        global semeado com SEED.

    Returns
    -------
    list[dict]
        Até n submissões, ordenadas pelo bucket (estratégia de solução direta
        primeiro, com mais tentativas por último).
    """
    if rng is None:
        rng = random.Random(SEED)

    buckets: dict[int, list[dict]] = {1: [], 2: [], 3: [], 4: [], 5: []}
    for event in correct_events:
        b = _bucket_for(int(event["total_attempts"]))
        buckets[b].append(event)

    sampled = []
    for b in (1, 2, 3, 4, 5):
        if len(sampled) >= n:
            break
        if buckets[b]:
            sampled.append(rng.choice(buckets[b]))
    return sampled


# Carrega submissões corretas e suas representações em código Java.
correct_by_assignment = load_correct_samples(SEQUENCES_PATH, CODE_STATES_PATH)

# Aplica diversity sampling por (assignment, problem) — RNG semeado para reproduzibilidade.
rng = random.Random(SEED)
sampled_codes: dict[int, dict[int, list[str]]] = {}
sampling_stats: list[dict] = []

for assignment_id in ASSIGNMENTS:
    sampled_codes[assignment_id] = {}
    per_problem = correct_by_assignment.get(assignment_id, {})
    for problem_id, events in per_problem.items():
        sampled = diversity_sample(events, n=5, rng=rng)
        sampled_codes[assignment_id][problem_id] = [s["code"] for s in sampled]
        bucket_counts = Counter(_bucket_for(s["total_attempts"]) for s in sampled)
        sampling_stats.append({
            "AssignmentID": assignment_id,
            "ProblemID": problem_id,
            "n_correct_total": len(events),
            "n_sampled": len(sampled),
            "buckets_used": sorted(bucket_counts.keys()),
        })

stats_df = pd.DataFrame(sampling_stats).sort_values(["AssignmentID", "ProblemID"]).reset_index(drop=True)

summary = (
    stats_df.groupby("AssignmentID")
    .agg(
        n_problems=("ProblemID", "nunique"),
        mean_samples=("n_sampled", "mean"),
        min_samples=("n_sampled", "min"),
        max_samples=("n_sampled", "max"),
        mean_correct_pool=("n_correct_total", "mean"),
    )
    .round(2)
)

print("Resumo do diversity sampling por assignment:")
print(summary.to_string())
print()
print(f"Total de problemas cobertos: {stats_df['ProblemID'].count()} "
      f"(soma de problemas únicos por assignment, com possível overlap entre assignments).")
print(f"Média global de amostras/problema: {stats_df['n_sampled'].mean():.2f}")
print(f"Distribuição do número de amostras por problema: "
      f"{dict(Counter(stats_df['n_sampled']).most_common())}")

Resumo do diversity sampling por assignment:
              n_problems  mean_samples  min_samples  max_samples  mean_correct_pool
AssignmentID                                                                       
439                   10           4.9            4            5              216.6
487                   10           4.9            4            5              177.3
492                   10           5.0            5            5              176.9
494                   10           5.0            5            5              180.1
502                   10           4.9            4            5              184.5

Total de problemas cobertos: 50 (soma de problemas únicos por assignment, com possível overlap entre assignments).
Média global de amostras/problema: 4.94
Distribuição do número de amostras por problema: {5: 47, 4: 3}


**Achado:** Os 5 assignments do Release/Train (A439, A487, A492, A494, A502) são processados em sua totalidade — `n_problems` por assignment está reportado em `summary`. A média de amostras/problema é ≤5 e tipicamente próxima de 5 nos problemas com maior pool de submissões corretas; problemas raros, com poucos acertos, geram menos amostras (1 por bucket disponível). A distribuição de `n_sampled` está concentrada em 5 — comportamento esperado para uma turma de ~233 estudantes em problemas de granularidade média.

**Implicação para modelagem:** O dicionário `sampled_codes` é o input direto da Etapa 2 (geração de KCs por LLM). Mantemos `n=5` porque Duan et al. (2025, Table 5) demonstram saturação acima desse número (AUC de 0.811 com n=7 e 0.810 com n=10 vs 0.812 com n=5) — aumentar `n` adicionaria custo LLM sem ganho mensurável na qualidade dos KCs. A estratificação por tentativas garante que problemas com soluções fáceis (todos acertam de primeira) ainda sejam representados, mesmo quando o pool de buckets 2–5 é vazio. Esta escolha é uma decisão arquitetural local: Duan et al. amostram aleatoriamente; nossa estratégia prioriza diversidade explícita de estratégia, hipótese a ser validada na Etapa 7 (post-hoc, via assinaturas srcML).

### 2.1 — Geração de KCs por LLM (Etapa 2)

**Contexto:** O CSEDM não fornece enunciados textuais dos problemas — apenas snapshots de código dos alunos. Para inferir os Knowledge Components (KCs) de cada problema, o pipeline KCGen-KT envia as submissões corretas amostradas diretamente ao LLM, que raciocina em cadeia: código → descrição do problema → KCs necessários. Esta etapa produz um dicionário bruto `{problem_id: {description, kcs}}` por assignment, que será consolidado na Etapa 3 (clustering Sentence-BERT + HAC). O cache em `results/kc_raw_{assignment_id}.json` torna o notebook re-executável sem novas chamadas API.

**Hipótese:** A partir de 4–5 soluções corretas estratificadas por dificuldade, o LLM consegue inferir uma descrição semântica coerente do problema e 3–7 KCs de granularidade compatível com problemas introdutórios Java. O uso de in-context examples estabiliza o formato e a granularidade dos KCs, reduzindo a variabilidade entre problemas.

**Referência:** Duan et al. (2025), Appendix B (Table 8) — in-context examples para KC generation; Table 4 — ablação de representação de código: **código bruto → AUC 0.812** vs. Student Code + AST → 0.800 vs. AST only → 0.784. LLMs não são pré-treinados em XML de AST; enviar AST *piora* a qualidade dos KCs gerados. Decisão arquitetural: **LLM recebe código bruto (não AST).** Ablação sem in-context examples: AUC 0.782 (−3pp) vs. base 0.812 — exemplos são obrigatórios.

In [3]:
# In-context examples adapted from Duan et al. (2025), Appendix B (Table 8).
# Two introductory Java problems at CSEDM granularity: array sum and grade mapping.
# These examples anchor KC naming convention (3-8 words) and granularity (3-7 KCs/problem).
_FEW_SHOT_EXAMPLES = """\
=== FEW-SHOT EXAMPLE 1 ===
Problem A — Correct student solutions (n=3):

Solution 1:
```java
public static int sumArray(int[] arr) {
    int total = 0;
    for (int i = 0; i < arr.length; i++) {
        total += arr[i];
    }
    return total;
}
```
Solution 2:
```java
public static int sumArray(int[] arr) {
    int sum = 0;
    for (int num : arr) {
        sum += num;
    }
    return sum;
}
```
Solution 3:
```java
public static int sumArray(int[] numbers) {
    int result = 0;
    int i = 0;
    while (i < numbers.length) {
        result += numbers[i];
        i++;
    }
    return result;
}
```
Expected JSON output:
{
  "problem_description": "Compute the sum of all integer elements in an array and return the total.",
  "kcs": [
    {"name": "Array traversal with loop", "reasoning": "All solutions iterate through every element using index-based for, enhanced for-each, or while loops."},
    {"name": "Accumulator variable pattern", "reasoning": "Each solution initializes a running total to zero and adds each element incrementally."},
    {"name": "Loop boundary with array length", "reasoning": "Index-based solutions use arr.length as the exclusive upper bound to avoid index out-of-bounds."},
    {"name": "Method return statement", "reasoning": "The method must return the computed integer, exercising the return keyword with a non-void type."}
  ]
}

=== FEW-SHOT EXAMPLE 2 ===
Problem B — Correct student solutions (n=3):

Solution 1:
```java
public static String letterGrade(int score) {
    if (score >= 90) return "A";
    else if (score >= 80) return "B";
    else if (score >= 70) return "C";
    else if (score >= 60) return "D";
    else return "F";
}
```
Solution 2:
```java
public static String letterGrade(int score) {
    String g;
    if (score >= 90) { g = "A"; }
    else if (score >= 80) { g = "B"; }
    else if (score >= 70) { g = "C"; }
    else if (score >= 60) { g = "D"; }
    else { g = "F"; }
    return g;
}
```
Solution 3:
```java
public static String letterGrade(int s) {
    if (s >= 90) return "A";
    if (s >= 80) return "B";
    if (s >= 70) return "C";
    if (s >= 60) return "D";
    return "F";
}
```
Expected JSON output:
{
  "problem_description": "Map a numeric score to a letter grade (A/B/C/D/F) using threshold-based conditionals.",
  "kcs": [
    {"name": "Multi-branch conditional chaining", "reasoning": "All solutions use if/else-if chains (or sequential ifs with early return) to partition the score into grade buckets."},
    {"name": "Numeric comparison with threshold", "reasoning": "Each branch tests the score against a fixed boundary (90, 80, 70, 60) using >= to select the grade."},
    {"name": "String return value from method", "reasoning": "The method returns a String literal, requiring correct non-void method return type declaration."},
    {"name": "Default case residual handling", "reasoning": "All solutions correctly handle scores below 60 as the final else/return without an explicit threshold check."}
  ]
}
"""

_SYSTEM_PROMPT = (
    "You are an expert CS educator specializing in introductory Java programming courses. "
    "Analyze correct student solutions to infer what Knowledge Components (KCs) are required "
    "to solve the problem. KCs must be: (a) specific and teachable (3-8 words each), "
    "(b) abstract enough to generalize across solutions, (c) 3-7 KCs per problem. "
    "Reason step-by-step: first infer what the problem asks from the code patterns, "
    "then identify the specific KCs demonstrated across the solutions."
)


def _build_kc_prompt(problem_id: int, code_samples: list[str]) -> str:
    """Construct the chain-of-thought prompt with in-context examples (Table 8)."""
    solutions = "".join(
        f"\nSolution {i}:\n```java\n{code}\n```"
        for i, code in enumerate(code_samples, 1)
    )
    return (
        f"{_FEW_SHOT_EXAMPLES}\n"
        "=== NOW ANALYZE THE FOLLOWING ===\n\n"
        f"Problem {problem_id} — Correct student solutions (n={len(code_samples)}):"
        f"{solutions}\n\n"
        "Reason step-by-step, then respond ONLY with a valid JSON object:\n"
        "{\n"
        '  "problem_description": "1-2 sentence description of what this problem requires",\n'
        '  "kcs": [\n'
        '    {"name": "KC name (3-8 words)", "reasoning": "Why this KC is needed (1 sentence)"},\n'
        "    ...\n"
        "  ]\n"
        "}"
    )


def generate_kcs_for_problem(
    problem_id: int,
    code_samples: list[str],
    client: anthropic.Anthropic,
) -> dict:
    """Generate KCs for one problem from raw Java code (not AST).

    Input: raw student code. Decision: Duan et al. (2025) Table 4 ablation shows
    AST degrades AUC 0.812 → 0.784; LLMs are not pre-trained on XML.

    Returns
    -------
    dict
        {"problem_description": str, "kcs": [{"name": str, "reasoning": str}, ...]}
    """
    raw = client.messages.create(
        model=LLM_MODEL,
        max_tokens=1024,
        system=_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": _build_kc_prompt(problem_id, code_samples)}],
    ).content[0].text.strip()

    # Strip markdown code fences if model wraps the JSON
    if "```json" in raw:
        raw = raw.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in raw:
        raw = raw.split("```", 1)[1].split("```", 1)[0].strip()

    # Find the outermost JSON object in case model adds preamble text
    start = raw.find("{")
    end = raw.rfind("}") + 1
    result = json.loads(raw[start:end])

    assert "problem_description" in result and "kcs" in result, f"Unexpected structure: {list(result.keys())}"
    assert all("name" in kc and "reasoning" in kc for kc in result["kcs"]), "KC missing name/reasoning"
    return result

In [4]:
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

kc_raw_all: dict[int, dict[int, dict]] = {}

for assignment_id in ASSIGNMENTS:
    cache_path = RESULTS_ROOT / f"kc_raw_A{assignment_id}.json"

    if cache_path.exists():
        print(f"A{assignment_id}: cache hit — loading {cache_path.name}")
        with open(cache_path) as f:
            kc_raw_all[assignment_id] = {int(k): v for k, v in json.load(f).items()}
        continue

    print(f"A{assignment_id}: calling LLM ({len(sampled_codes[assignment_id])} problems) ...")
    assignment_result: dict[int, dict] = {}

    for problem_id, codes in sorted(sampled_codes[assignment_id].items()):
        result = generate_kcs_for_problem(problem_id, codes, client)
        assignment_result[problem_id] = result
        n_kcs = len(result["kcs"])
        desc_preview = result["problem_description"][:70]
        print(f"  P{problem_id}: {n_kcs} KCs — {desc_preview}")

    kc_raw_all[assignment_id] = assignment_result

    # Persist immediately — notebook re-runs load cache without new API calls
    with open(cache_path, "w") as f:
        json.dump({str(k): v for k, v in assignment_result.items()}, f, indent=2, ensure_ascii=False)
    print(f"  → saved {cache_path.name}")

# Summary
print("\n--- KC Generation Summary ---")
for aid in ASSIGNMENTS:
    probs = kc_raw_all.get(aid, {})
    if not probs:
        print(f"  A{aid}: NOT LOADED")
        continue
    n_probs = len(probs)
    total_kcs = sum(len(v["kcs"]) for v in probs.values())
    print(f"  A{aid}: {n_probs} problems, {total_kcs} raw KCs ({total_kcs / n_probs:.1f} avg/problem)")

# Verify all cache files were written
missing = [aid for aid in ASSIGNMENTS if not (RESULTS_ROOT / f"kc_raw_A{aid}.json").exists()]
assert not missing, f"Cache files missing for assignments: {missing}"
print("\nAll kc_raw_A*.json cache files present.")

A439: cache hit — loading kc_raw_A439.json
A487: cache hit — loading kc_raw_A487.json
A492: cache hit — loading kc_raw_A492.json
A494: cache hit — loading kc_raw_A494.json
A502: cache hit — loading kc_raw_A502.json

--- KC Generation Summary ---
  A439: 10 problems, 58 raw KCs (5.8 avg/problem)
  A487: 10 problems, 55 raw KCs (5.5 avg/problem)
  A492: 10 problems, 59 raw KCs (5.9 avg/problem)
  A494: 10 problems, 60 raw KCs (6.0 avg/problem)
  A502: 10 problems, 60 raw KCs (6.0 avg/problem)

All kc_raw_A*.json cache files present.


**Achado:** KCs brutos gerados para todos os 5 assignments (A439, A487, A492, A494, A502), cobrindo 50 problemas no total. A contagem média de KCs por problema e a descrição inferida do problema estão reportadas no sumário acima. Os arquivos `results/kc_raw_A*.json` são persistidos após a primeira execução — execuções subsequentes (incluindo o `verify_cmd` do harness) carregam do cache sem novas chamadas LLM.

**Implicação para modelagem:** Os KCs brutos desta etapa são o input direto da Etapa 3 (clustering Sentence-BERT + HAC), que consolida o vocabulário de ~50 KCs brutos por assignment em 10–15 KCs canônicos (Q-matrix). A granularidade desta geração (3–7 KCs por problema, 3–8 palavras por nome) fundamenta toda a interpretabilidade downstream do pipeline KCGen-KT — KCs excessivamente genéricos ("programming skill") ou específicos demais ("use of variable named total") degradam tanto o clustering quanto o correctness labeling da Etapa 6.

### 3.1 — Clustering Sentence-BERT + HAC dos KCs brutos (Etapa 3)

**Contexto:** A Etapa 2 gerou ~55–60 KCs brutos por assignment — um por chamada LLM, com sobreposição semântica inevitável (e.g., "Compound boolean condition with AND operator" e "Logical AND for range inclusion" expressam o mesmo conceito). Para construir uma Q-matrix com 10–15 KCs canônicos, os KCs brutos precisam ser consolidados por similaridade semântica. Embeddings de sentença (Sentence-BERT) mapeiam os nomes dos KCs para um espaço vetorial denso onde proximidade captura similaridade de conceito; HAC com distância cosseno agrupa KCs semanticamente próximos sem exigir pré-especificação de centroides.

**Hipótese:** Com ~55–60 KCs brutos por assignment (5–6 por problema, 10 problemas), um agrupamento de 10–15 clusters captura a granularidade média de KCs adequada para problemas introdutórios Java — proporcional aos 50 clusters para 50 problemas encontrados por Duan et al. (2025). A seleção do melhor `n_clusters` dentro do intervalo {10, 12, 15} via silhouette score garante que o número escolhido maximize a coesão intra-cluster e a separação inter-cluster para cada assignment.

**Referência:** Duan et al. (2025) — Seção 3.1.2: clustering HAC com sentence-transformers para consolidação de KCs brutos; resultado: abstração média (1 cluster ≈ 1 problema) supera alta abstração (poucos clusters abrangentes). Rousseeuw (1987) — silhouette score como critério de seleção de `k`.

In [5]:
from sentence_transformers import SentenceTransformer
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.metrics import silhouette_score

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
N_CLUSTERS_CANDIDATES = [10, 12, 15]


def _cluster_with_n(embeddings: np.ndarray, n_clusters: int) -> np.ndarray:
    """HAC with cosine distance (average linkage), returns 0-indexed labels."""
    dist_condensed = pdist(embeddings, metric="cosine")
    dist_condensed = np.clip(dist_condensed, 0, None)  # fix floating-point negatives
    Z = linkage(dist_condensed, method="average")
    labels = fcluster(Z, t=n_clusters, criterion="maxclust")
    return labels - 1  # scipy fcluster is 1-indexed


def select_best_n_clusters(
    embeddings: np.ndarray,
    candidates: list[int],
) -> tuple[int, dict[int, float]]:
    """Select n_clusters maximising average silhouette score (Rousseeuw, 1987).

    Silhouette score measures cohesion vs. separation: higher is better (range [-1, 1]).
    Candidates where n >= n_unique_kcs are skipped (would produce singleton clusters).
    """
    n_unique = len(embeddings)
    scores: dict[int, float] = {}
    for n in candidates:
        if n >= n_unique:
            continue
        labels = _cluster_with_n(embeddings, n)
        if len(set(labels)) < 2:
            scores[n] = -1.0
            continue
        scores[n] = float(silhouette_score(embeddings, labels, metric="cosine"))
    best_n = max(scores, key=lambda k: scores[k])
    return best_n, scores


# Load model once (downloaded on first run, cached locally by sentence-transformers)
print(f"Loading embedding model: {EMBEDDING_MODEL}")
encoder = SentenceTransformer(EMBEDDING_MODEL)
np.random.seed(SEED)  # reproducibility for any internal randomness

kc_clusters_all: dict[int, dict] = {}

for assignment_id in ASSIGNMENTS:
    cache_path = RESULTS_ROOT / f"kc_clusters_A{assignment_id}.json"

    if cache_path.exists():
        print(f"A{assignment_id}: cache hit — loading {cache_path.name}")
        with open(cache_path) as f:
            kc_clusters_all[assignment_id] = json.load(f)
        continue

    print(f"A{assignment_id}: computing embeddings and HAC clustering ...")

    # Collect unique KC names for this assignment (order-preserving dedup)
    seen: set[str] = set()
    unique_kc_names: list[str] = []
    for problem_id in sorted(kc_raw_all[assignment_id].keys()):
        for kc in kc_raw_all[assignment_id][problem_id]["kcs"]:
            name = kc["name"]
            if name not in seen:
                unique_kc_names.append(name)
                seen.add(name)

    n_raw = sum(len(v["kcs"]) for v in kc_raw_all[assignment_id].values())
    print(f"  {n_raw} raw KC entries → {len(unique_kc_names)} unique KC names")

    # Sentence-BERT embeddings (normalized for cosine distance)
    embeddings = encoder.encode(
        unique_kc_names, normalize_embeddings=True, show_progress_bar=False
    )

    # Select best n_clusters from {10, 12, 15} by silhouette score
    best_n, sil_scores = select_best_n_clusters(embeddings, N_CLUSTERS_CANDIDATES)
    print(f"  silhouette scores: { {k: round(v, 4) for k, v in sil_scores.items()} }")
    print(f"  → selected n_clusters = {best_n}")

    # Final clustering with selected n
    labels = _cluster_with_n(embeddings, best_n)

    kc_to_cluster: dict[str, int] = {
        name: int(lbl) for name, lbl in zip(unique_kc_names, labels)
    }
    clusters: dict[str, list[str]] = {}
    for cluster_id in range(best_n):
        members = [n for n, lbl in kc_to_cluster.items() if lbl == cluster_id]
        if members:
            clusters[str(cluster_id)] = members

    result = {
        "assignment_id": f"A{assignment_id}",
        "n_clusters_selected": best_n,
        "silhouette_scores": {str(k): round(v, 6) for k, v in sil_scores.items()},
        "kc_to_cluster": kc_to_cluster,
        "clusters": clusters,
    }
    kc_clusters_all[assignment_id] = result

    with open(cache_path, "w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"  → saved {cache_path.name}")

# --- Summary ---
print("\n--- Clustering Summary ---")
for aid in ASSIGNMENTS:
    res = kc_clusters_all.get(aid, {})
    n_sel = res.get("n_clusters_selected")
    n_kcs = len(res.get("kc_to_cluster", {}))
    sil = res.get("silhouette_scores", {})
    print(f"  A{aid}: {n_kcs} unique KCs → {n_sel} clusters | silhouette: {sil}")

missing = [aid for aid in ASSIGNMENTS if not (RESULTS_ROOT / f"kc_clusters_A{aid}.json").exists()]
assert not missing, f"Cache files missing for assignments: {missing}"
print("\nAll kc_clusters_A*.json cache files present.")

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11511.45it/s]

A439: cache hit — loading kc_clusters_A439.json
A487: cache hit — loading kc_clusters_A487.json
A492: cache hit — loading kc_clusters_A492.json
A494: cache hit — loading kc_clusters_A494.json
A502: cache hit — loading kc_clusters_A502.json

--- Clustering Summary ---
  A439: 52 unique KCs → 15 clusters | silhouette: {'10': 0.297947, '12': 0.308072, '15': 0.323346}
  A487: 55 unique KCs → 15 clusters | silhouette: {'10': 0.196762, '12': 0.2043, '15': 0.262111}
  A492: 58 unique KCs → 15 clusters | silhouette: {'10': 0.193745, '12': 0.225509, '15': 0.228313}
  A494: 59 unique KCs → 15 clusters | silhouette: {'10': 0.199154, '12': 0.201797, '15': 0.222199}
  A502: 59 unique KCs → 12 clusters | silhouette: {'10': 0.166204, '12': 0.179088, '15': 0.15461}

All kc_clusters_A*.json cache files present.


**Achado:** Todos os 5 assignments foram clusterizados com sucesso. O silhouette score selecionou `n_clusters=15` para 4 de 5 assignments (A439: 0.323, A487: 0.262, A492: 0.228, A494: 0.222) e `n_clusters=12` para A502 (silhouette 0.179 em 12 vs. 0.155 em 15 — gradiente de qualidade invertido, indicando que os KCs de A502 são naturalmente mais compactos em 12 grupos). Os scores estão na faixa 0.15–0.32, tipica de embeddings semânticos com fronteiras difusas entre conceitos próximos (e.g., "Logical AND" e "Compound boolean condition" residem em regiões vizinhas no espaço de embedding). O número de KCs únicos após deduplicação nominal é 52–59 por assignment (de ~55–60 brutos), indicando sobreposição mínima de nomes exatos — a maioria das redundâncias é semântica, não textual, e portanto capturada corretamente pelo clustering.

**Implicação para modelagem:** Os arquivos `results/kc_clusters_A*.json` fornecem o mapeamento `kc_name → cluster_id` que a Etapa 4 (rotulagem via LLM) usará para gerar os rótulos canônicos dos clusters. Com 12–15 clusters por assignment para 10 problemas, a granularidade é compatível com a recomendação de Duan et al. (2025) de ~1 cluster por problema como alvo de abstração média. Assignments com silhouette baixo (A502: 0.18) indicam KCs altamente interdependentes — comportamento esperado em problemas que exigem múltiplas skills simultâneas — e não é um problema para o pipeline, pois a Q-matrix representa associações, não partições exclusivas.

### 4.1 — Rotulagem de clusters via LLM (Etapa 4)

**Contexto:** A Etapa 3 produziu 12–15 clusters por assignment, cada um contendo vários KCs brutos semanticamente próximos. Para construir a Q-matrix, cada cluster precisa de um rótulo canônico único — o KC final. Um segundo prompt LLM recebe todos os nomes do cluster e decide: (a) se um KC existente já representa o grupo adequadamente, ou (b) se é necessário sintetizar um novo rótulo que capture o conceito comum. O cache em `results/kc_descriptions_{assignment_id}.json` garante re-executabilidade sem novas chamadas API.

**Hipótese:** A maioria dos clusters terá um KC dominante que já captura bem o conceito central; clusters heterogêneos (e.g., mistura de "iteração indexada" e "iteração por for-each") exigirão síntese de um novo rótulo mais abstrato. O LLM consegue fazer essa distinção com um prompt chain-of-thought simples.

**Referência:** Duan et al. (2025), Stage 3 (Table 9) — prompt de rotulagem de clusters via LLM com instrução de decisão explícita (selecionar KC existente vs. sintetizar novo rótulo). Adaptado diretamente ao contexto Java introdutório do CSEDM.

In [6]:
# Prompt adapted from Duan et al. (2025), Table 9 — cluster labeling stage.
# Citation: "We prompt the LLM to either select the most representative KC from the
# cluster or synthesize a new label capturing the shared concept." — Section 3.1.2.
_CLUSTER_LABEL_SYSTEM = (
    "You are an expert CS educator labeling clusters of Knowledge Components (KCs) "
    "for an introductory Java programming course. "
    "A KC is a specific, teachable concept (3-8 words). "
    "Your task: given a cluster of semantically similar KCs, decide whether one KC "
    "already represents the group well, or synthesize a concise new label that captures "
    "the common underlying concept."
)


def label_cluster(cluster_kcs: list[str], client: anthropic.Anthropic, cluster_id: int) -> dict:
    """Label one cluster of KCs using LLM (Duan et al. 2025, Table 9 prompt).

    Parameters
    ----------
    cluster_kcs : list[str]
        KC names belonging to this cluster (from Sentence-BERT HAC, Etapa 3).
    client : anthropic.Anthropic
        Initialized Anthropic client.
    cluster_id : int
        0-indexed cluster identifier; embedded verbatim in the returned dict.

    Returns
    -------
    dict
        {"kc_id": int, "name": str, "reasoning": str}
        kc_id matches cluster_id; name is 3-8 words; reasoning is 1 sentence.
    """
    kc_list = "\n".join(f"- {kc}" for kc in cluster_kcs)

    prompt = (
        "You are labeling a cluster of related Knowledge Components for a programming course.\n\n"
        "The following KCs were grouped together based on semantic similarity:\n"
        f"{kc_list}\n\n"
        "Decide: does one of these KCs already represent the entire cluster well, "
        "or should you synthesize a new label that captures the common underlying concept?\n\n"
        f"Respond ONLY with a valid JSON object (cluster_id = {cluster_id}):\n"
        "{{\n"
        f'  "kc_id": {cluster_id},\n'
        '  "name": "Final KC label (3-8 words)",\n'
        '  "reasoning": "Why this label represents the cluster (1 sentence)"\n'
        "}}"
    )

    raw = client.messages.create(
        model=LLM_MODEL,
        max_tokens=256,
        system=_CLUSTER_LABEL_SYSTEM,
        messages=[{"role": "user", "content": prompt}],
    ).content[0].text.strip()

    # Strip markdown code fences if present
    if "```json" in raw:
        raw = raw.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in raw:
        raw = raw.split("```", 1)[1].split("```", 1)[0].strip()

    start = raw.find("{")
    end = raw.rfind("}") + 1
    result = json.loads(raw[start:end])
    result["kc_id"] = cluster_id  # guarantee correct id regardless of LLM output
    return result


# --- Cache-aware labeling loop ---
kc_descriptions_all: dict[str, list[dict]] = {}

for assignment_id in ASSIGNMENTS:
    aid_str = f"A{assignment_id}"
    cache_path = RESULTS_ROOT / f"kc_descriptions_{aid_str}.json"

    if cache_path.exists():
        print(f"{aid_str}: cache hit — loading {cache_path.name}")
        with open(cache_path) as f:
            kc_descriptions_all[aid_str] = json.load(f)
        continue

    clusters_path = RESULTS_ROOT / f"kc_clusters_{aid_str}.json"
    with open(clusters_path) as f:
        clusters_data = json.load(f)

    clusters: dict[str, list[str]] = clusters_data["clusters"]
    descriptions: list[dict] = []

    print(f"{aid_str}: labeling {len(clusters)} clusters via LLM ...")
    for cluster_id_str, kc_names in clusters.items():
        cluster_id = int(cluster_id_str)
        desc = label_cluster(kc_names, client, cluster_id)
        descriptions.append(desc)
        print(f"  cluster {cluster_id:02d}: {desc['name']}")

    descriptions.sort(key=lambda x: x["kc_id"])

    with open(cache_path, "w") as f:
        json.dump(descriptions, f, indent=2, ensure_ascii=False)
    kc_descriptions_all[aid_str] = descriptions
    print(f"  → saved {cache_path.name} ({len(descriptions)} KCs)")

# --- Summary ---
print("\n--- Cluster Labeling Summary ---")
for aid in ASSIGNMENTS:
    aid_str = f"A{aid}"
    descs = kc_descriptions_all.get(aid_str, [])
    print(f"  {aid_str}: {len(descs)} canonical KCs")

missing = [aid for aid in ASSIGNMENTS
           if not (RESULTS_ROOT / f"kc_descriptions_A{aid}.json").exists()]
assert not missing, f"Cache files missing for assignments: {missing}"
print("\nAll kc_descriptions_A*.json cache files present.")

A439: cache hit — loading kc_descriptions_A439.json
A487: cache hit — loading kc_descriptions_A487.json
A492: cache hit — loading kc_descriptions_A492.json
A494: cache hit — loading kc_descriptions_A494.json
A502: cache hit — loading kc_descriptions_A502.json

--- Cluster Labeling Summary ---
  A439: 15 canonical KCs
  A487: 15 canonical KCs
  A492: 15 canonical KCs
  A494: 15 canonical KCs
  A502: 12 canonical KCs

All kc_descriptions_A*.json cache files present.


In [7]:
from IPython.display import Markdown, display

# Display canonical KCs for A439 as a formatted table
aid_display = "A439"
descs_439 = kc_descriptions_all[aid_display]

rows = ["| KC ID | Nome canônico | Raciocínio |",
        "|------:|:--------------|:-----------|"]
for d in descs_439:
    rows.append(f"| {d['kc_id']} | {d['name']} | {d['reasoning']} |")

display(Markdown(f"#### KCs canônicos — {aid_display} ({len(descs_439)} clusters)\n\n" + "\n".join(rows)))

#### KCs canônicos — A439 (15 clusters)

| KC ID | Nome canônico | Raciocínio |
|------:|:--------------|:-----------|
| 0 | Range validation with logical operators | This label captures the core concept that all KCs share: using comparison and logical operators together to check if values fall within specified numeric ranges. |
| 1 | Numeric threshold comparison with operators | This KC already clearly and specifically captures the concept of using comparison operators to evaluate numeric values against threshold limits. |
| 2 | Equality comparison of numeric values | This label subsumes the more specific 'integers' case and generalizes to all numeric types (int, double, float, long), making it the most comprehensive representation of the cluster. |
| 3 | Absolute value for comparison operations | This label clarifies that absolute value is used as a computational tool for performing comparisons rather than just calculating magnitude. |
| 4 | Boolean negation with NOT operator | This label most comprehensively captures the cluster by naming both the concept (Boolean negation) and the specific operator (!), avoiding redundancy while remaining concise. |
| 5 | Inequality operators for negative conditions | This KC is the only item in the cluster and already provides a specific, teachable label for the concept of using inequality operators (!=, <, >, <=, >=) to express negation or opposite conditions. |
| 6 | Conditional branching with multiple paths | This label captures the shared concept of using conditionals to handle multiple decision branches, whether through nesting, chaining, or logical operators. |
| 7 | Using arithmetic operations in conditionals | This label captures both the evaluation of arithmetic expressions and their use within conditional statements as a unified concept. |
| 8 | Compound boolean expressions with logical operators | This label captures the core concept encompassing both AND and OR logical operators used to combine multiple conditions in boolean expressions, which is the common thread across all cluster items. |
| 9 | Returning values from conditional statements | This label captures the common concept of using return statements within if-else blocks to output different values based on conditions, encompassing both the control flow and value-return aspects. |
| 10 | Early return pattern in control flow | This label captures the common concept of using early returns to exit methods based on conditional logic, encompassing nested blocks, short-circuiting, and method control flow without being unnecessarily specific. |
| 11 | Method return types and values | This label captures the common concept of understanding different data types (boolean, integer) that methods can return and work with. |
| 12 | Integer return value with discrete states | This single KC already captures the specific concept of using integer return values to represent distinct program states or conditions, making it a sufficient label for the cluster. |
| 13 | Method parameter usage in computation | This single KC already precisely captures the concept of using method parameters within computational logic and operations. |
| 14 | Default case residual handling | This single KC already precisely describes the concept of handling remaining cases in switch statements when default case is used, making it an adequate cluster label. |

**Achado:** Todos os 5 assignments receberam rótulos canônicos para seus clusters (12–15 KCs por assignment). O número final de KCs canônicos é igual ao `n_clusters_selected` da Etapa 3 — nenhum cluster foi descartado. Os rótulos para A439 estão exibidos na tabela acima; a coluna "Raciocínio" registra a justificativa do LLM para cada escolha (seleção de KC existente vs. síntese de novo rótulo). Os arquivos `results/kc_descriptions_A*.json` são persistidos após a primeira execução — execuções subsequentes carregam do cache sem novas chamadas LLM.

**Implicação para modelagem:** Os KCs canônicos desta etapa formam o vocabulário da Q-matrix (Etapa 5): cada linha da Q-matrix será um `ProblemID` e cada coluna um dos 12–15 KCs canônicos do assignment. A granularidade 12–15 KCs / 10 problemas é consistente com a recomendação de Duan et al. (2025) de abstração média — nem demasiado fino (um KC por skill específica) nem demasiado grosseiro (poucos KCs para muitos problemas). Na Etapa 6, esses mesmos rótulos serão usados como referência no prompt de correctness labeling, que associa cada submissão incorreta aos KCs que o estudante falhou em demonstrar.